# ChemicalEntity ↔ Gene Relation-Wise Merge


## 0. Configuration

In [1]:
import pandas as pd
import numpy as np
import re

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CHEMICALENTITY_GENE/ALL_CHEMICALENTITY_GENE.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. Chemical — PubChem and DrugBank

In [2]:
Pubchem = pd.read_pickle(DB_DIR + 'pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem
print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict: 119,177,440 entries


In [3]:
Drugbank = pd.read_csv(DB_DIR + 'drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))
print(f"DrugBank dict: {len(Drugbank_dict):,} entries")

DrugBank dict: 16,575 entries


In [4]:
#

### 1b. Gene — NCBI and ENSEMBL

In [5]:
# ENSEMBL ID → NCBI Symbol mapping
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

# Full NCBI gene table
NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)

NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))
NCBI_ALL_Symb_Desc_dict  = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))

# Exploded synonym → canonical Symbol dict
NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = v

print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")
print(f"Synonym dict:    {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} entries")

NCBI gene table: 193,505 rows
Synonym dict:    69,564 entries


## 2. Load KG Sources

### 2a. DRKG

In [6]:
DRKG_Chemical_Gene = pd.read_csv(PROC_DIR + 'DRKG/DRKG_ChemicalEntity_Gene.csv')
DRKG_Chemical_Gene.columns = DRKG_Chemical_Gene.columns.str.lower()
print(f"DRKG:      {len(DRKG_Chemical_Gene):,} rows | columns: {list(DRKG_Chemical_Gene.columns)}")


DRKG_Chemical_Gene['kg_type'] = 'Generalised'
DRKG_Chemical_Gene['kg_source'] = 'DRKG'
DRKG_Chemical_Gene['species'] = 'Homo sapiens'

DRKG_Chemical_Gene.head(2)

DRKG:      115,233 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id', 'tail_detail_name', 'tail_ens', 'head_id_is', 'tail_id_is', 'head_orignal', 'tail_orignal']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id,tail_detail_name,tail_ens,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,448286,Compound_Gene,RNASE1,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,DB02573,"ribonuclease A family member 1, pancreatic",ENSG00000129538,Pubchem,NCBI_ID,Compound::DB02573,Gene::6035,Generalised,Homo sapiens
1,28871,Compound_Gene,PTGDR2,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,DB13217,prostaglandin D2 receptor 2,ENSG00000183134,Pubchem,NCBI_ID,Compound::DB13217,Gene::11251,Generalised,Homo sapiens


### 2b. CKG (×3)

In [7]:
CKG_Chemical_Gene_1 = pd.read_csv(PROC_DIR + 'CKG/File_30_Drug_Gene_CKG.csv')
CKG_Chemical_Gene_1.columns = CKG_Chemical_Gene_1.columns.str.lower()
CKG_Chemical_Gene_1 = CKG_Chemical_Gene_1[~CKG_Chemical_Gene_1['head'].isna()]
CKG_Chemical_Gene_1['head'] = CKG_Chemical_Gene_1['head'].astype(str).str.replace(r'\.0$', '', regex=True)

CKG_Chemical_Gene_1['kg_type'] = 'Generalised'
CKG_Chemical_Gene_1['kg_source'] = 'CKG'
CKG_Chemical_Gene_1['species'] = 'Homo sapiens'



CKG_Chemical_Gene_2 = pd.read_csv(PROC_DIR + 'CKG/File_1_Drug_Gene_CKG.csv')
CKG_Chemical_Gene_2.columns = CKG_Chemical_Gene_2.columns.str.lower()
CKG_Chemical_Gene_2.rename(columns={'tail_name': 'tail_detail_name'}, inplace=True)

CKG_Chemical_Gene_2['kg_type'] = 'Generalised'
CKG_Chemical_Gene_2['kg_source'] = 'CKG'
CKG_Chemical_Gene_2['species'] = 'Homo sapiens'


CKG_Chemical_Gene_3 = pd.read_csv(PROC_DIR + 'CKG/File_4_Drug_Gene_OncoKB_CKG.csv')
CKG_Chemical_Gene_3.columns = CKG_Chemical_Gene_3.columns.str.lower()

CKG_Chemical_Gene_3['kg_type'] = 'Generalised'
CKG_Chemical_Gene_3['kg_source'] = 'CKG'
CKG_Chemical_Gene_3['species'] = 'Homo sapiens'


print(f"CKG 1:     {len(CKG_Chemical_Gene_1):,} rows")
print(f"CKG 2:     {len(CKG_Chemical_Gene_2):,} rows")
print(f"CKG 3:     {len(CKG_Chemical_Gene_3):,} rows")

CKG 1:     799 rows
CKG 2:     20,292 rows
CKG 3:     111 rows


### 2c. PharmKG

In [8]:
PharmKG_Chemical_Gene = pd.read_csv(PROC_DIR + 'PharmKG/PharmKG_Chemical_Gene.csv')
PharmKG_Chemical_Gene.columns = PharmKG_Chemical_Gene.columns.str.lower()
print(f"PharmKG:   {len(PharmKG_Chemical_Gene):,} rows | columns: {list(PharmKG_Chemical_Gene.columns)}")

PharmKG_Chemical_Gene['kg_type'] = 'Generalised'
PharmKG_Chemical_Gene['kg_source'] = 'PharmKG'
PharmKG_Chemical_Gene['species'] = 'Homo sapiens'

PharmKG_Chemical_Gene.head(2)

PharmKG:   85,707 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'tail_detail_name', 'tail_alias_mapped', 'head_id_is', 'tail_id_is', 'head_orignal', 'pubmed_id', 'sentence_tokenized']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,tail_detail_name,tail_alias_mapped,head_id_is,tail_id_is,head_orignal,pubmed_id,sentence_tokenized,kg_type,species
0,4004,Chemical_Gene,AR,chemical,E,gene,PharmKG,amphiregulin,AREG,Pubchem,NCBI_ID,mlt,26295055.0,'MLT reduced the immunostained cells for andro...,Generalised,Homo sapiens
1,5789,Chemical_Gene,SLC28A3,chemical,Gene-Chemical,gene,PharmKG,solute carrier family 28 member 3,SLC28A3,Pubchem,NCBI_ID,deoxythymidine,NaN,NaN,Generalised,Homo sapiens


### 2d. Hetionet

In [9]:
Hetionet_Chemical_Gene = pd.read_csv(PROC_DIR + 'Hetionet/Compound_Gene_Hetionet.csv')
Hetionet_Chemical_Gene.columns = Hetionet_Chemical_Gene.columns.str.lower()
print(f"Hetionet:  {len(Hetionet_Chemical_Gene):,} rows | columns: {list(Hetionet_Chemical_Gene.columns)}")

Hetionet_Chemical_Gene['kg_type'] = 'Generalised'
Hetionet_Chemical_Gene['species'] = 'Homo sapiens'

Hetionet_Chemical_Gene.head(2)

Hetionet:  51,428 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_pubchem_mapped', 'tail_name', 'tail_detail_name']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_pubchem_mapped,tail_name,tail_detail_name,kg_type,species
0,5360696,Compound_Gene,CHRNA3,Compound,Compound–binds–Gene,Gene,Hetionet,Pubchem,NCBI_ID,DB00514,1136,cholinergic receptor nicotinic alpha 3 subunit,Generalised,Homo sapiens
1,37720,Compound_Gene,FGF1,Compound,Compound–binds–Gene,Gene,Hetionet,Pubchem,NCBI_ID,DB00686,2246,fibroblast growth factor 1,Generalised,Homo sapiens


### 2e. TARKG

In [10]:
TARKG_Chemical_Gene = pd.read_csv(PROC_DIR + 'TARKG/Compound_Gene.csv')
TARKG_Chemical_Gene.columns = TARKG_Chemical_Gene.columns.str.lower()
print(f"TARKG:     {len(TARKG_Chemical_Gene):,} rows | columns: {list(TARKG_Chemical_Gene.columns)}")
TARKG_Chemical_Gene['kg_type'] = 'Generalised'
TARKG_Chemical_Gene['species'] = 'Homo sapiens'

TARKG_Chemical_Gene.head(2)

TARKG:     633,243 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_index', 'kg', 'change']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,123134263,ChemicalEntity_Gene,TOP2A,ChemicalEntity,negative_correlate,Gene,TARKG,"(3R,6R,9R,15R,18R,21R,24R,30R)-30-ethyl-33-[(E...",DNA topoisomerase II alpha,Pubchem,NCBI_ID,addKG-11489059,addKG,0,Generalised,Homo sapiens
1,145068,ChemicalEntity_Gene,TOP2A,ChemicalEntity,negative_correlate,Gene,TARKG,nitric oxide,DNA topoisomerase II alpha,Pubchem,NCBI_ID,addKG-13142344,addKG,0,Generalised,Homo sapiens


### 2f. iBKH

Filter out non-PubChem/DrugBank head prefixes; fix swapped `tail_detail_name` columns.

In [11]:
iBKH_Chemical_Gene = pd.read_csv(PROC_DIR + 'iBKH/Chemical_Gene_ibkh.csv')
iBKH_Chemical_Gene.columns = iBKH_Chemical_Gene.columns.str.lower()
# Drop rows with non-standard head ID prefixes
iBKH_Chemical_Gene = iBKH_Chemical_Gene[
    ~iBKH_Chemical_Gene['head'].str.startswith(('PharmGKB', 'KEG', 'MES', 'MeS', 'PA1', 'PA4'))
]
# Fix swapped name columns
# iBKH_Chemical_Gene[['tail_detail_name', 'tail_detail_name.1']] = \
#     iBKH_Chemical_Gene[['tail_detail_name.1', 'tail_detail_name']]
iBKH_Chemical_Gene['kg_type'] = 'Generalised'
iBKH_Chemical_Gene['species'] = 'Homo sapiens'

print(f"iBKH:      {len(iBKH_Chemical_Gene):,} rows | columns: {list(iBKH_Chemical_Gene.columns)}")
iBKH_Chemical_Gene.head(2)

iBKH:      566,779 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_type', 'species']


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,1051,ChemicalEntity_Gene,HDC,ChemicalEntity,Gene,iBKH,(4-formyl-5-hydroxy-6-methylpyridin-3-yl)methy...,CC1=NC=C(C(=C1O)C=O)COP(=O)(O)O,histidine decarboxylase,Pubchem,NCBI_ID,Generalised,Homo sapiens
1,6274,ChemicalEntity_Gene,HDC,ChemicalEntity,Gene,iBKH,(2S)-2-amino-3-(1H-imidazol-5-yl)propanoic acid,C1=C(NC=N1)C[C@@H](C(=O)O)N,histidine decarboxylase,Pubchem,NCBI_ID,Generalised,Homo sapiens


In [12]:
iBKH_Chemical_Gene_2 = pd.read_csv(PROC_DIR + 'iBKH/Drug_Gene_ibkh.csv')
iBKH_Chemical_Gene_2.columns = iBKH_Chemical_Gene_2.columns.str.lower()

print(f"iBKH:      {len(iBKH_Chemical_Gene_2):,} rows | columns: {list(iBKH_Chemical_Gene_2.columns)}")
iBKH_Chemical_Gene_2['kg_type'] = 'Generalised'
iBKH_Chemical_Gene_2['species'] = 'Homo sapiens'

iBKH_Chemical_Gene_2.head(2)

iBKH:      554,236 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,1051,ChemicalEntity_Gene,HDC,ChemicalEntity,Gene,iBKH,(4-formyl-5-hydroxy-6-methylpyridin-3-yl)methy...,CC1=NC=C(C(=C1O)C=O)COP(=O)(O)O,histidine decarboxylase,Pubchem,NCBI_ID,Generalised,Homo sapiens
1,6274,ChemicalEntity_Gene,HDC,ChemicalEntity,Gene,iBKH,(2S)-2-amino-3-(1H-imidazol-5-yl)propanoic acid,C1=C(NC=N1)C[C@@H](C(=O)O)N,histidine decarboxylase,Pubchem,NCBI_ID,Generalised,Homo sapiens


### 2g. hald

In [13]:
hald = pd.read_csv(PROC_DIR + 'hald/Chemical_Gene_HALD.csv')
hald.columns = hald.columns.str.lower()
print(f"hald:    \ {len(hald):,} rows | columns: {list(hald.columns)}")

hald['kg_type'] = 'Aging'
hald['species'] = 'Homo sapiens'

hald.head(2)

hald:    \ 709 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,118984454,ChemicalEntity_Gene,INS,ChemicalEntity,result,Gene,HALD_KG,"(4S)-4-[[2-[[(1R,6R,12S,15S,18S,21S,24S,27S,30...",CC[C@H](C)[C@H]1C(=O)N[C@H]2CSSC[C@@H](C(=O)N[...,insulin,Pubchem,NCBI_ID,Aging,Homo sapiens
1,Ceramides,ChemicalEntity_Gene,SLC2A4,ChemicalEntity,reduce,Gene,HALD_KG,Ceramides,Ceramides,solute carrier family 2 member 4,NaN,NCBI_ID,Aging,Homo sapiens


### 2h. Evolf

In [14]:
Evolf_Chemical_Gene = pd.read_csv(PROC_DIR + 'evolf/Chemical_Gene_Evolf.csv')
Evolf_Chemical_Gene.columns = Evolf_Chemical_Gene.columns.str.lower()
print(f"Evolf:     {len(Evolf_Chemical_Gene):,} rows | columns: {list(Evolf_Chemical_Gene.columns)}")

Evolf_Chemical_Gene['kg_type'] = 'Generalised'
Evolf_Chemical_Gene['species'] = 'Homo sapiens'


Evolf_Chemical_Gene.head(2)

Evolf:     81,925 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smile_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,62329,ChemicalEntity_Gene,OR8B8,ChemicalEntity,Gene,Evolf,"1-tert-butyl-3,5-dimethyl-2,4,6-trinitrobenzene",CC1=C(C(=C(C(=C1[N+](=O)[O-])C(C)(C)C)[N+](=O)...,olfactory receptor family 8 subfamily B member 8,Pubchem,NCBI_ID,Generalised,Homo sapiens
1,62329,ChemicalEntity_Gene,OR8B12,ChemicalEntity,Gene,Evolf,"1-tert-butyl-3,5-dimethyl-2,4,6-trinitrobenzene",CC1=C(C(=C(C(=C1[N+](=O)[O-])C(C)(C)C)[N+](=O)...,olfactory receptor family 8 subfamily B member 12,Pubchem,NCBI_ID,Generalised,Homo sapiens


### 2i. Biosnap sms chembl

In [15]:
chembl = pd.read_csv(PROC_DIR + 'chembl/Chemical_Gene_Human_Chembl_Metaboglue.csv')
chembl.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
chembl.columns = chembl.columns.str.lower()
chembl['kg_type'] = 'Generalised'
chembl['kg_source'] = 'Chembl'

chembl['species'] = 'Homo sapiens'

chembl


,head,relation,tail,head_type,tail_type,kg_source,species,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type
0,44342104,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl,Homo sapiens,"6-(4-bromoanilino)-7-chloroquinazoline-5,8-dione",C1=CC(=CC=C1NC2=C(C(=O)C3=NC=NC=C3C2=O)Cl)Br,DNA topoisomerase I,Pubchem,NCBI_ID,Generalised
1,10405313,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl,Homo sapiens,"7-chloro-6-(2,4-dimethoxyanilino)quinazoline-5...",COC1=CC(=C(C=C1)NC2=C(C(=O)C3=NC=NC=C3C2=O)Cl)OC,DNA topoisomerase I,Pubchem,NCBI_ID,Generalised
2,5472243,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl,Homo sapiens,"(11E)-11-(4-chlorobutylidene)-15,16-dimethoxy-...",CN1C2=C(C3=CC(=C(C=C3C1=O)OC)OC)/C(=C/CCCCl)/C...,DNA topoisomerase I,Pubchem,NCBI_ID,Generalised
3,10244294,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl,Homo sapiens,2-(3H-benzimidazol-5-yl)-3H-benzimidazole-5-ca...,C1=CC2=C(C=C1C#N)NC(=N2)C3=CC4=C(C=C3)N=CN4,DNA topoisomerase I,Pubchem,NCBI_ID,Generalised
4,44340518,ChemicalEntity_Gene,TOP1,ChemicalEntity,Gene,Chembl,Homo sapiens,"(11Z)-11-(4-iodobutylidene)-15,16-dimethoxy-20...",CN1C2=C(C3=CC(=C(C=C3C1=O)OC)OC)/C(=C\CCCI)/C4...,DNA topoisomerase I,Pubchem,NCBI_ID,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...
77797,4371240,ChemicalEntity_Gene,LIMK1,ChemicalEntity,Gene,Chembl,Homo sapiens,N-benzyl-4-(benzylsulfamoyl)benzamide,C1=CC=C(C=C1)CNC(=O)C2=CC=C(C=C2)S(=O)(=O)NCC3...,LIM domain kinase 1,Pubchem,NCBI_ID,Generalised
77798,137796294,ChemicalEntity_Gene,LIMK1,ChemicalEntity,Gene,Chembl,Homo sapiens,N-benzyl-N-butyl-4-(phenylsulfamoyl)benzamide,CCCCN(CC1=CC=CC=C1)C(=O)C2=CC=C(C=C2)S(=O)(=O)...,LIM domain kinase 1,Pubchem,NCBI_ID,Generalised
77799,137796294,ChemicalEntity_Gene,LIMK1,ChemicalEntity,Gene,Chembl,Homo sapiens,N-benzyl-N-butyl-4-(phenylsulfamoyl)benzamide,CCCCN(CC1=CC=CC=C1)C(=O)C2=CC=C(C=C2)S(=O)(=O)...,LIM domain kinase 1,Pubchem,NCBI_ID,Generalised
77800,137796294,ChemicalEntity_Gene,LIMK1,ChemicalEntity,Gene,Chembl,Homo sapiens,N-benzyl-N-butyl-4-(phenylsulfamoyl)benzamide,CCCCN(CC1=CC=CC=C1)C(=O)C2=CC=C(C=C2)S(=O)(=O)...,LIM domain kinase 1,Pubchem,NCBI_ID,Generalised


In [16]:
biosnap = pd.read_csv(PROC_DIR + 'biosnap/Chemical_Gene_BioSNAP_2_Metaboglue.csv')
biosnap.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
biosnap.columns = biosnap.columns.str.lower()
biosnap['relation']   = 'ChemicalEntity_Gene'
biosnap['head_type']  = 'ChemicalEntity'
biosnap['tail_type']  = 'Gene'

biosnap['kg_type'] = 'Generalised'
biosnap['kg_source'] = 'BioSNAP'
biosnap['species'] = 'Homo sapiens'
biosnap


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,100003355,ChemicalEntity_Gene,KCNH2,ChemicalEntity,Gene,BioSNAP,"diethyl (2S,3R)-2-ethylsulfonyl-3-(4-methoxyph...",CCOC(=O)C1([C@H]([C@@H]1S(=O)(=O)CC)C2=CC=C(C=...,potassium voltage-gated channel subfamily H me...,Pubchem,NCBI_ID,Generalised,Homo sapiens
1,100003355,ChemicalEntity_Gene,SCN5A,ChemicalEntity,Gene,BioSNAP,"diethyl (2S,3R)-2-ethylsulfonyl-3-(4-methoxyph...",CCOC(=O)C1([C@H]([C@@H]1S(=O)(=O)CC)C2=CC=C(C=...,sodium voltage-gated channel alpha subunit 5,Pubchem,NCBI_ID,Generalised,Homo sapiens
2,100003510,ChemicalEntity_Gene,KCNH2,ChemicalEntity,Gene,BioSNAP,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,potassium voltage-gated channel subfamily H me...,Pubchem,NCBI_ID,Generalised,Homo sapiens
3,100003510,ChemicalEntity_Gene,HTR3A,ChemicalEntity,Gene,BioSNAP,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,5-hydroxytryptamine receptor 3A,Pubchem,NCBI_ID,Generalised,Homo sapiens
4,100003510,ChemicalEntity_Gene,HTR7,ChemicalEntity,Gene,BioSNAP,"[(1R,2R,3R)-1-amino-2-ethylsulfonyl-3-(4-metho...",CCS(=O)(=O)[C@@H]1[C@@H]([C@]1(CO)N)C2=CC=C(C=...,5-hydroxytryptamine receptor 7,Pubchem,NCBI_ID,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
65291,100125889,ChemicalEntity_Gene,CACNA2D4,ChemicalEntity,Gene,BioSNAP,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,calcium voltage-gated channel auxiliary subuni...,Pubchem,NCBI_ID,Generalised,Homo sapiens
65292,100125889,ChemicalEntity_Gene,CACNA2D2,ChemicalEntity,Gene,BioSNAP,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,calcium voltage-gated channel auxiliary subuni...,Pubchem,NCBI_ID,Generalised,Homo sapiens
65293,100125889,ChemicalEntity_Gene,CACNA2D3,ChemicalEntity,Gene,BioSNAP,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,calcium voltage-gated channel auxiliary subuni...,Pubchem,NCBI_ID,Generalised,Homo sapiens
65294,100125889,ChemicalEntity_Gene,CACNA2D1,ChemicalEntity,Gene,BioSNAP,"N-[(3-chlorophenyl)methyl]-4-[2,3-dihydro-1,4-...",CC(C)CNC(=O)[C@@H](CC1=CC=CC=C1)N(CC2=CC(=CC=C...,calcium voltage-gated channel auxiliary subuni...,Pubchem,NCBI_ID,Generalised,Homo sapiens


In [17]:
sms = pd.read_csv(PROC_DIR + 'sms/Chemical_Gene_Human_SMS_Metaboglue.csv')
sms.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
sms.columns = sms.columns.str.lower()
sms['kg_type'] = 'Generalised'
sms['species'] = 'Homo sapiens'
sms


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,6380,ChemicalEntity_Gene,CHRNA4,ChemicalEntity,Gene,SMS,tetramethylazanium,C[N+](C)(C)C,cholinergic receptor nicotinic alpha 4 subunit,Pubchem,NCBI_ID,Generalised,Homo sapiens
1,6380,ChemicalEntity_Gene,CHRNA7,ChemicalEntity,Gene,SMS,tetramethylazanium,C[N+](C)(C)C,cholinergic receptor nicotinic alpha 7 subunit,Pubchem,NCBI_ID,Generalised,Homo sapiens
2,6380,ChemicalEntity_Gene,CHRNB2,ChemicalEntity,Gene,SMS,tetramethylazanium,C[N+](C)(C)C,cholinergic receptor nicotinic beta 2 subunit,Pubchem,NCBI_ID,Generalised,Homo sapiens
3,6380,ChemicalEntity_Gene,CHRFAM7A,ChemicalEntity,Gene,SMS,tetramethylazanium,C[N+](C)(C)C,CHRNA7 (exons 5-10) and FAM7A (exons A-E) fusion,Pubchem,NCBI_ID,Generalised,Homo sapiens
4,892,ChemicalEntity_Gene,APP,ChemicalEntity,Gene,SMS,"cyclohexane-1,2,3,4,5,6-hexol",C1(C(C(C(C(C1O)O)O)O)O)O,amyloid beta precursor protein,Pubchem,NCBI_ID,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1277074,155567712,ChemicalEntity_Gene,PSMB8,ChemicalEntity,Gene,SMS,"(2S)-N-[(2S,3R)-1-[[(E,2S)-1-[4-(aminomethyl)p...",C[C@H]([C@@H](C(=O)N[C@@H](CC1=CC=C(C=C1)CN)/C...,proteasome 20S subunit beta 8,Pubchem,NCBI_ID,Generalised,Homo sapiens
1277075,155567712,ChemicalEntity_Gene,PSMB9,ChemicalEntity,Gene,SMS,"(2S)-N-[(2S,3R)-1-[[(E,2S)-1-[4-(aminomethyl)p...",C[C@H]([C@@H](C(=O)N[C@@H](CC1=CC=C(C=C1)CN)/C...,proteasome 20S subunit beta 9,Pubchem,NCBI_ID,Generalised,Homo sapiens
1277076,155567712,ChemicalEntity_Gene,PSMB9,ChemicalEntity,Gene,SMS,"(2S)-N-[(2S,3R)-1-[[(E,2S)-1-[4-(aminomethyl)p...",C[C@H]([C@@H](C(=O)N[C@@H](CC1=CC=C(C=C1)CN)/C...,proteasome 20S subunit beta 9,Pubchem,NCBI_ID,Generalised,Homo sapiens
1277077,155567712,ChemicalEntity_Gene,PSMB10,ChemicalEntity,Gene,SMS,"(2S)-N-[(2S,3R)-1-[[(E,2S)-1-[4-(aminomethyl)p...",C[C@H]([C@@H](C(=O)N[C@@H](CC1=CC=C(C=C1)CN)/C...,proteasome 20S subunit beta 10,Pubchem,NCBI_ID,Generalised,Homo sapiens


In [18]:
print(f"file: {len(chembl):,} rows")
print(f"file: {len(biosnap):,} rows")
print(f"file: {len(sms):,} rows")

file: 77,802 rows
file: 65,296 rows
file: 1,277,079 rows


### 2j Ageanno

In [19]:
ageanno = pd.read_csv(PROC_DIR + 'ageanno_processed/AgeAnno_Chem_Gene.csv')
# ageanno.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
ageanno.columns = ageanno.columns.str.lower()
ageanno['kg_source'] = 'AgeAnno'
ageanno['kg_type'] = 'Aging'
ageanno['species'] = 'Homo sapiens'
ageanno

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_smile_name,relation_source,head_id_is,tail_id_is,interactionactions,pubmedids,species,kg_source,kg_type
0,2836600,ChemicalEntity_Gene,AR,ChemicalEntity,Gene,"4-nitro-N-(2-phenylphenyl)-2,1,3-benzoxadiazol...",androgen receptor,C1=CC=C(C=C1)C2=CC=CC=C2NC3=CC=C(C4=NON=C34)[N...,AgeAnno,Pubchem,NCBI_ID,10074-G5 inhibits the reaction [EPHB2 protein ...,32184358,Homo sapiens,AgeAnno,Aging
1,2836600,ChemicalEntity_Gene,AR,ChemicalEntity,Gene,"4-nitro-N-(2-phenylphenyl)-2,1,3-benzoxadiazol...",androgen receptor,C1=CC=C(C=C1)C2=CC=CC=C2NC3=CC=C(C4=NON=C34)[N...,AgeAnno,Pubchem,NCBI_ID,10074-G5 results in decreased expression of AR...,32184358,Homo sapiens,AgeAnno,Aging
2,2836600,ChemicalEntity_Gene,AR,ChemicalEntity,Gene,"4-nitro-N-(2-phenylphenyl)-2,1,3-benzoxadiazol...",androgen receptor,C1=CC=C(C=C1)C2=CC=CC=C2NC3=CC=C(C4=NON=C34)[N...,AgeAnno,Pubchem,NCBI_ID,10074-G5 results in decreased expression of AR...,32184358,Homo sapiens,AgeAnno,Aging
3,2836600,ChemicalEntity_Gene,EPHB2,ChemicalEntity,Gene,"4-nitro-N-(2-phenylphenyl)-2,1,3-benzoxadiazol...",EPH receptor B2,C1=CC=C(C=C1)C2=CC=CC=C2NC3=CC=C(C4=NON=C34)[N...,AgeAnno,Pubchem,NCBI_ID,10074-G5 inhibits the reaction [EPHB2 protein ...,32184358,Homo sapiens,AgeAnno,Aging
4,2836600,ChemicalEntity_Gene,EPHB2,ChemicalEntity,Gene,"4-nitro-N-(2-phenylphenyl)-2,1,3-benzoxadiazol...",EPH receptor B2,C1=CC=C(C=C1)C2=CC=CC=C2NC3=CC=C(C4=NON=C34)[N...,AgeAnno,Pubchem,NCBI_ID,10074-G5 inhibits the reaction [EPHB2 protein ...,32184358,Homo sapiens,AgeAnno,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382460,11647372,ChemicalEntity_Gene,PIK3CB,ChemicalEntity,Gene,4-[4-[2-(difluoromethyl)benzimidazol-1-yl]-6-m...,"phosphatidylinositol-4,5-bisphosphate 3-kinase...",C1COCCN1C2=NC(=NC(=N2)N3C4=CC=CC=C4N=C3C(F)F)N...,AgeAnno,Pubchem,NCBI_ID,ZSTK474 analog results in decreased activity o...,21882832,Homo sapiens,AgeAnno,Aging
382461,11647372,ChemicalEntity_Gene,PIK3CB,ChemicalEntity,Gene,4-[4-[2-(difluoromethyl)benzimidazol-1-yl]-6-m...,"phosphatidylinositol-4,5-bisphosphate 3-kinase...",C1COCCN1C2=NC(=NC(=N2)N3C4=CC=CC=C4N=C3C(F)F)N...,AgeAnno,Pubchem,NCBI_ID,ZSTK474 results in decreased activity of PIK3C...,20129775,Homo sapiens,AgeAnno,Aging
382462,11647372,ChemicalEntity_Gene,PIK3CG,ChemicalEntity,Gene,4-[4-[2-(difluoromethyl)benzimidazol-1-yl]-6-m...,"phosphatidylinositol-4,5-bisphosphate 3-kinase...",C1COCCN1C2=NC(=NC(=N2)N3C4=CC=CC=C4N=C3C(F)F)N...,AgeAnno,Pubchem,NCBI_ID,ZSTK474 results in decreased activity of PIK3C...,20129775,Homo sapiens,AgeAnno,Aging
382463,11647372,ChemicalEntity_Gene,PRKDC,ChemicalEntity,Gene,4-[4-[2-(difluoromethyl)benzimidazol-1-yl]-6-m...,"protein kinase, DNA-activated, catalytic subunit",C1COCCN1C2=NC(=NC(=N2)N3C4=CC=CC=C4N=C3C(F)F)N...,AgeAnno,Pubchem,NCBI_ID,ZSTK474 results in decreased activity of PRKDC...,20129775,Homo sapiens,AgeAnno,Aging


## 2k. Pheknowlator

In [20]:
pheknowlator = pd.read_csv(PROC_DIR + 'pheknowlator/Chemical_Gene.csv')
pheknowlator.columns = pheknowlator.columns.str.lower()
pheknowlator['kg_source'] = 'pheknowlator'
pheknowlator['kg_type'] = 'Generalised'
pheknowlator['species'] = 'Homo sapiens'
pheknowlator['head_type'] = 'ChemicalEntity'
pheknowlator['relation'] = 'ChemicalEntity_Gene'
pheknowlator['head_detail_name'] = pheknowlator['head'].map(Pubchem_IUPAC_CID_Dict)
pheknowlator

,head,relation,tail,head_type,tail_type,head_iupac,head_smiles,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type,species,head_detail_name
0,2256,ChemicalEntity_Gene,EXOC5,ChemicalEntity,Gene,NaN,NaN,exocyst complex component 5,Pubchem_ID,NCBI_ID,pheknowlator,Generalised,Homo sapiens,NaN
1,445639,ChemicalEntity_Gene,LPIN1,ChemicalEntity,Gene,NaN,NaN,lipin 1,Pubchem_ID,NCBI_ID,pheknowlator,Generalised,Homo sapiens,NaN
2,3121,ChemicalEntity_Gene,ADCY2,ChemicalEntity,Gene,NaN,NaN,adenylate cyclase 2,Pubchem_ID,NCBI_ID,pheknowlator,Generalised,Homo sapiens,NaN
3,31703,ChemicalEntity_Gene,LAMA5,ChemicalEntity,Gene,NaN,NaN,laminin subunit alpha 5,Pubchem_ID,NCBI_ID,pheknowlator,Generalised,Homo sapiens,NaN
4,9064,ChemicalEntity_Gene,DEDD,ChemicalEntity,Gene,NaN,NaN,death effector domain containing,Pubchem_ID,NCBI_ID,pheknowlator,Generalised,Homo sapiens,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223554,5591,ChemicalEntity_Gene,TOP2A,ChemicalEntity,Gene,NaN,NaN,DNA topoisomerase II alpha,Pubchem_ID,NCBI_ID,pheknowlator,Generalised,Homo sapiens,NaN
223555,6758,ChemicalEntity_Gene,LINC01698,ChemicalEntity,Gene,NaN,NaN,long intergenic non-protein coding RNA 1698,Pubchem_ID,NCBI_ID,pheknowlator,Generalised,Homo sapiens,NaN
223556,5641,ChemicalEntity_Gene,TNFAIP3,ChemicalEntity,Gene,NaN,NaN,TNF alpha induced protein 3,Pubchem_ID,NCBI_ID,pheknowlator,Generalised,Homo sapiens,NaN
223557,9823887,ChemicalEntity_Gene,MC1R,ChemicalEntity,Gene,NaN,NaN,melanocortin 1 receptor,Pubchem_ID,NCBI_ID,pheknowlator,Generalised,Homo sapiens,NaN


In [21]:
#

## 3. Check and Fix Duplicate Columns

In [22]:
SOURCE_DFS = [
    ('DRKG_Chemical_Gene',    DRKG_Chemical_Gene),
    ('CKG_Chemical_Gene_1',   CKG_Chemical_Gene_1),
    ('CKG_Chemical_Gene_2',   CKG_Chemical_Gene_2),
    ('CKG_Chemical_Gene_3',   CKG_Chemical_Gene_3),
    ('PharmKG_Chemical_Gene', PharmKG_Chemical_Gene),
    ('Hetionet_Chemical_Gene',Hetionet_Chemical_Gene),
    ('TARKG_Chemical_Gene',   TARKG_Chemical_Gene),
    ('iBKH_Chemical_Gene',    iBKH_Chemical_Gene),
    ('iBKH_Chemical_Gene_2', iBKH_Chemical_Gene_2),
    ('hald',                hald),  # aging
    ('ageanno',           ageanno),  # aging
    ('Evolf_Chemical_Gene',   Evolf_Chemical_Gene),
    ('chembl',           chembl),
    ('biosnap',           biosnap),
    ('sms',           sms),
    ('pheknowlator', pheknowlator),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[DRKG_Chemical_Gene] ✓ no duplicates
[CKG_Chemical_Gene_1] ✓ no duplicates
[CKG_Chemical_Gene_2] ✓ no duplicates
[CKG_Chemical_Gene_3] ✓ no duplicates
[PharmKG_Chemical_Gene] ✓ no duplicates
[Hetionet_Chemical_Gene] ✓ no duplicates
[TARKG_Chemical_Gene] ✓ no duplicates
[iBKH_Chemical_Gene] ✓ no duplicates
[iBKH_Chemical_Gene_2] ✓ no duplicates
[hald] ✓ no duplicates
[ageanno] ✓ no duplicates
[Evolf_Chemical_Gene] ✓ no duplicates
[chembl] ✓ no duplicates
[biosnap] ✓ no duplicates
[sms] ✓ no duplicates
[pheknowlator] ✓ no duplicates


## 4. Align Schemas and Concatenate

In [23]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 4,136,663 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,448286,Compound_Gene,RNASE1,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,Pubchem,NCBI_ID,NaN,"ribonuclease A family member 1, pancreatic",Generalised,Homo sapiens
1,28871,Compound_Gene,PTGDR2,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,Pubchem,NCBI_ID,NaN,prostaglandin D2 receptor 2,Generalised,Homo sapiens


In [24]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,3229141,3229141,6458282
tail_type,0,0,0
kg_source,0,0,0
head_id_is,203,203,406
tail_id_is,0,0,0
head_detail_name,510347,510347,1020694


In [25]:
final_df[final_df['head_id_is'].isna()]  # valid 
final_df[final_df['head_detail_name'].isna()]


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,448286,Compound_Gene,RNASE1,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,Pubchem,NCBI_ID,NaN,"ribonuclease A family member 1, pancreatic",Generalised,Homo sapiens
1,28871,Compound_Gene,PTGDR2,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,Pubchem,NCBI_ID,NaN,prostaglandin D2 receptor 2,Generalised,Homo sapiens
2,1948,Compound_Gene,MMP3,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,Pubchem,NCBI_ID,NaN,matrix metallopeptidase 3,Generalised,Homo sapiens
3,23586154,Compound_Gene,AFDN,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,Pubchem,NCBI_ID,NaN,"afadin, adherens junction formation factor",Generalised,Homo sapiens
4,6857692,Compound_Gene,F11,Compound,bioarx::DrugHumGen:Compound:Gene,Gene,DRKG,Pubchem,NCBI_ID,NaN,coagulation factor XI,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4136658,5591,ChemicalEntity_Gene,TOP2A,ChemicalEntity,NaN,Gene,pheknowlator,Pubchem_ID,NCBI_ID,NaN,DNA topoisomerase II alpha,Generalised,Homo sapiens
4136659,6758,ChemicalEntity_Gene,LINC01698,ChemicalEntity,NaN,Gene,pheknowlator,Pubchem_ID,NCBI_ID,NaN,long intergenic non-protein coding RNA 1698,Generalised,Homo sapiens
4136660,5641,ChemicalEntity_Gene,TNFAIP3,ChemicalEntity,NaN,Gene,pheknowlator,Pubchem_ID,NCBI_ID,NaN,TNF alpha induced protein 3,Generalised,Homo sapiens
4136661,9823887,ChemicalEntity_Gene,MC1R,ChemicalEntity,NaN,Gene,pheknowlator,Pubchem_ID,NCBI_ID,NaN,melanocortin 1 receptor,Generalised,Homo sapiens


## 5. Standardise Fixed-Value Columns

In [26]:
#

In [27]:
final_df['head_type'] = 'ChemicalEntity'
final_df['relation']  = 'ChemicalEntity_Gene'
final_df['tail_type'] = 'Gene'
final_df['tail_id_is'] = 'NCBI_ID'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_type:",  set(final_df['head_type']))
print("Unique tail_type:",  set(final_df['tail_type']))
print("Unique kg_source:",  set(final_df['kg_source']))
print("Unique kg_type:",  set(final_df['kg_type']))


Unique relation: {'ChemicalEntity_Gene'}
Unique head_type: {'ChemicalEntity'}
Unique tail_type: {'Gene'}
Unique kg_source: {'iBKH', 'HALD_KG', 'Chembl', 'BioSNAP', 'TARKG', 'Evolf', 'SMS', 'AgeAnno', 'pheknowlator', 'DRKG', 'Hetionet', 'PharmKG', 'CKG'}
Unique kg_type: {'Aging', 'Generalised'}


## 6. Deduplicate

In [28]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 3,007,449 rows


In [29]:
print("Unique relation:",   set(final_df_dedup['relation']))
print("Unique head_type:",  set(final_df_dedup['head_type']))
print("Unique tail_type:",  set(final_df_dedup['tail_type']))
print("Unique kg_source:",  set(final_df_dedup['kg_source']))
print("Unique kg_type:",  set(final_df_dedup['kg_type']))

Unique relation: {'ChemicalEntity_Gene'}
Unique head_type: {'ChemicalEntity'}
Unique tail_type: {'Gene'}
Unique kg_source: {'iBKH', 'PharmKG::SMS::iBKH', 'Chembl::SMS::TARKG::iBKH', 'Evolf', 'Evolf::PharmKG::TARKG', 'AgeAnno::SMS::TARKG::iBKH::pheknowlator', 'AgeAnno::PharmKG::SMS::TARKG::pheknowlator', 'pheknowlator', 'Evolf::PharmKG::TARKG::iBKH::pheknowlator', 'CKG::DRKG::HALD_KG::Hetionet::iBKH', 'BioSNAP', 'AgeAnno::Evolf::iBKH::pheknowlator', 'Evolf::SMS::pheknowlator', 'AgeAnno::Chembl::PharmKG::TARKG::iBKH::pheknowlator', 'Chembl::PharmKG::iBKH', 'Chembl::SMS', 'AgeAnno::Chembl::TARKG::pheknowlator', 'AgeAnno::Chembl::iBKH', 'PharmKG::pheknowlator', 'Evolf::SMS::iBKH::pheknowlator', 'CKG::DRKG', 'Chembl::SMS::TARKG', 'PharmKG::TARKG', 'AgeAnno::PharmKG::SMS', 'DRKG::HALD_KG::Hetionet::iBKH', 'PharmKG::SMS::iBKH::pheknowlator', 'PharmKG::SMS::TARKG::pheknowlator', 'Chembl::SMS::iBKH', 'Evolf::TARKG::iBKH', 'Evolf::TARKG', 'AgeAnno::PharmKG::SMS::TARKG', 'AgeAnno::SMS::iBKH', 'Ag

## 7. Repair Missing `tail_detail_name` (Gene Names)

1. Strip leading `-` from tail symbol.
2. Map through synonym dict to get canonical symbol.
3. Fall back to `NCBI_ALL_Symb_Desc_dict` (Symbol → description).

In [30]:
mask = final_df_dedup['tail_detail_name'].isna()
print(f"Rows needing tail_detail_name repair: {mask.sum():,}")

# Step 1 – clean symbol
final_df_dedup.loc[mask, 'tail'] = final_df_dedup.loc[mask, 'tail'].str.replace('-', '', regex=False)

# Step 2 – synonym → canonical symbol
final_df_dedup.loc[mask, 'tail'] = (
    final_df_dedup.loc[mask, 'tail']
    .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
    .fillna(final_df_dedup.loc[mask, 'tail'])
)

# Step 3 – NCBI Symbol → description
mask2 = final_df_dedup['tail_detail_name'].isna()
final_df_dedup.loc[mask2, 'tail_detail_name'] = final_df_dedup.loc[mask2, 'tail'].map(NCBI_ALL_Symb_Desc_dict)
print(f"Still missing after repair: {final_df_dedup['tail_detail_name'].isna().sum():,}")

Rows needing tail_detail_name repair: 594
Still missing after repair: 2


## 8. Resolve Head (Chemical) Names

1. PubChem IUPAC lookup (numeric CIDs).
2. Normalise DrugBank IDs to zero-padded 5-digit format.
3. DrugBank name fallback for `DB`-prefixed IDs.
4. Assign `head_id_is`: `DB`-prefix → `Drugbank`, else `Pubchem`.
5. Drop rows where either name could not be resolved.

In [31]:
# Step 1 – PubChem lookup
final_df_dedup['head_detail_name'] = final_df_dedup['head'].astype(str).map(Pubchem_IUPAC_CID_Dict)
final_df_dedup['head_smiles_name'] = final_df_dedup['head'].astype(str).map(Pubchem_CID_Smile_Dict)

# Step 2 – Normalise DrugBank IDs
def standardize_drugbank_id(val):
    if isinstance(val, str):
        m = re.match(r'^DB(\d+)$', val)
        if m:
            return 'DB' + m.group(1).zfill(5)
    return val

final_df_dedup['head'] = final_df_dedup['head'].apply(standardize_drugbank_id).astype(str)

# Step 3 – DrugBank fallback
mask = final_df_dedup['head_detail_name'].isna() & final_df_dedup['head'].str.startswith('DB')
final_df_dedup.loc[mask, 'head_detail_name'] = final_df_dedup.loc[mask, 'head'].map(Drugbank_dict)

# Step 4 – Assign head_id_is
final_df_dedup['head_id_is'] = np.where(
    final_df_dedup['head'].isna(), None,
    np.where(final_df_dedup['head'].str.startswith('DB'), 'Drugbank', 'Pubchem')
)

# Step 5 – Drop unresolvable rows
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[
    ~final_df_dedup['head_detail_name'].isna() &
    ~final_df_dedup['tail_detail_name'].isna()
].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} unresolvable rows → {len(final_df_dedup):,} remaining")

Dropped 2,807 unresolvable rows → 3,004,642 remaining


## 9. Add Schema Columns and Enforce Column Order

In [32]:
EXTRA_COLS = ['head_smiles_name']
final_df_dedup = final_df_dedup[REQUIRED_COLS + EXTRA_COLS]

print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (3004642, 14)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name
0,1,ChemicalEntity_Gene,APC,ChemicalEntity,associate,Gene,TARKG,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,APC regulator of WNT signaling pathway,Generalised,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C
1,1,ChemicalEntity_Gene,PNMT,ChemicalEntity,associate,Gene,TARKG,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,phenylethanolamine N-methyltransferase,Generalised,Homo sapiens,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C
2,3,ChemicalEntity_Gene,CD8A,ChemicalEntity,negative_correlate,Gene,TARKG,Pubchem,NCBI_ID,"5,6-dihydroxycyclohexa-1,3-diene-1-carboxylic ...",CD8 subunit alpha,Generalised,Homo sapiens,C1=CC(C(C(=C1)C(=O)O)O)O
3,3,ChemicalEntity_Gene,FOSL1,ChemicalEntity,negative_correlate,Gene,TARKG,Pubchem,NCBI_ID,"5,6-dihydroxycyclohexa-1,3-diene-1-carboxylic ...","FOS like 1, AP-1 transcription factor subunit",Generalised,Homo sapiens,C1=CC(C(C(=C1)C(=O)O)O)O
4,3,ChemicalEntity_Gene,GAPDH,ChemicalEntity,associate,Gene,TARKG,Pubchem,NCBI_ID,"5,6-dihydroxycyclohexa-1,3-diene-1-carboxylic ...",glyceraldehyde-3-phosphate dehydrogenase,Generalised,Homo sapiens,C1=CC(C(C(=C1)C(=O)O)O)O


## 10. QC — NaN Check

In [33]:
nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,2367367,0,2367367
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [34]:
#

## 11. Save Output

In [35]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 3,004,642 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CHEMICALENTITY_GENE/ALL_CHEMICALENTITY_GENE.csv


In [37]:
set(final_df_dedup['kg_type'])

{'Aging', 'Aging::Generalised', 'Generalised'}

In [1]:
# set(final_df_dedup['kg_source'])
